# xG — Copa do Mundo 2022

Modelo de *expected goals*: dado um chute, qual a probabilidade de virar gol.

Dados: [StatsBomb Open Data](https://github.com/statsbomb/open-data) — 64 partidas.

In [ ]:
import os, json, time
import requests

BASE = 'https://raw.githubusercontent.com/statsbomb/open-data/master/data'
COMPETICAO, TEMPORADA = 43, 106   # 43 = FIFA World Cup, 106 = 2022

CACHE = 'statsbomb_cache'
os.makedirs(CACHE, exist_ok=True)

In [ ]:
def baixar(caminho):
    """Baixa data/<caminho>. Se ja estiver em disco, le do cache."""
    local = os.path.join(CACHE, caminho.replace('/', '_'))
    if os.path.exists(local):
        return json.load(open(local, encoding='utf-8'))

    dados = requests.get(f'{BASE}/{caminho}', timeout=30).json()
    json.dump(dados, open(local, 'w', encoding='utf-8'))
    time.sleep(0.2)   # 64 requisicoes seguidas, nao vale martelar o servidor
    return dados


partidas = baixar(f'matches/{COMPETICAO}/{TEMPORADA}.json')
for p in partidas:
    baixar(f'events/{p["match_id"]}.json')

print(len(partidas), 'partidas no cache')

O cache bruto tem ~150 MB e fica fora do Git. O `extrair_chutes.py` percorre esses arquivos, filtra os eventos de chute e gera o `chutes_wc22.parquet` (0,12 MB), que é o que o resto do notebook usa.

In [2]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_parquet('chutes_wc22.parquet')
df.shape

(1494, 27)

In [4]:
df.head()

,match_id,shot_id,period,minute,second,x,y,duration,under_pressure,statsbomb_xg,...,shot_technique,shot_body_part,first_time,one_on_one,aerial_won,open_goal,deflected,follows_dribble,saved_to_post,gol
0,3857254,14374288-0565-4f12-b39e-318100137d0a,1,2,2,92.6,52.0,0.338598,False,0.023820,...,Normal,Right Foot,False,False,False,False,False,False,False,0
1,3857254,89cbea71-b5ef-4c16-9bd3-39b511619958,1,4,10,114.0,54.8,0.105592,False,0.014060,...,Normal,Left Foot,False,False,False,False,False,False,False,0
2,3857254,700d58f9-8032-4543-9a9d-b134c6996608,1,10,46,93.4,44.5,0.136106,False,0.033115,...,Normal,Right Foot,False,False,False,False,False,False,False,0
3,3857254,63422fb4-45a3-4f30-8340-53d468a8b8cb,1,11,47,114.7,29.6,0.970357,False,0.043661,...,Normal,Head,False,False,False,False,False,False,False,0
4,3857254,490287f7-4b99-49f8-ac89-16766ac7a05b,1,22,5,115.3,32.5,0.013816,False,0.124033,...,Volley,Right Foot,True,False,False,False,False,False,False,0
